# 0. Подготовка

Запускается **один раз**. Что делает:
1. переносит обученные модели HW1 в `hw2/artifacts/` под именами `hw1_<имя>.joblib` (+ копирует `dialog_flags.csv`);
2. собирает train/test и блоки кросс-валидации ровно как в HW1 (флаги `use_reaction`, группы `group_id`);
3. считает метрики старых моделей на кросс-валидации и записывает в `results/cv_table.csv`.

Дальше каждый в команде в своём ноутбуке запускает такую же ячейку подготовки и добавляет свою строку в ту же таблицу.

In [1]:
from pathlib import Path
import solution_utils as su
import pandas as pd

from solution_utils import CLASSES, SEED


## 1. Перенос моделей из первой домашки


`HW1_ARTIFACTS` — папка `artifacts` из HW1 (там лежат `*_1.joblib` и `dialog_flags.csv`).

Замечание: `kirp_model` в HW1 сохранена с токенайзер из ноутбука, и сама по себе без ноутбука не загрузится.
Перенесли дословно этот токенайзер в `solution_utils`.

In [2]:
HW1_ARTIFACTS = Path("../solution_hw1_BKZ/artifacts")     # <-- поправьте, если папка HW1 лежит в другом месте
assert HW1_ARTIFACTS.exists(), f"нет папки {HW1_ARTIFACTS}: укажите, где лежит artifacts/ из HW1"

moved = su.migrate_old_models(HW1_ARTIFACTS)
print("перенесено:", list(moved))

перенесено: ['hw1_most_frequent', 'hw1_stratified', 'hw1_bow_logreg', 'hw1_tfidf_logreg', 'hw1_kirp_model']


## 2. Данные и разбиение

In [3]:
# Задача 1 из HW1: диалоги с use_reaction == 1 (флаги из dialog_flags.csv), разбиение по group_id
task1 = su.load_task1(su.train_path(), su.flags_path())
train_df, test_df = su.split_task1(task1)          # то же train/test, что в HW1

train_dialogs = train_df[["dialog_id", "turns"]].to_dict("records")
test_dialogs = test_df[["dialog_id", "turns"]].to_dict("records")
y_train, y_test = train_df["reaction"].to_numpy(), test_df["reaction"].to_numpy()
folds = su.get_folds(y_train, train_df["group_id"].to_numpy())   # одни блоки CV для всех моделей

print(f"всего для задачи 1: {len(task1)} | train: {len(train_df)} | test: {len(test_df)}")
pd.DataFrame({"train": pd.Series(y_train).value_counts(normalize=True),
              "test": pd.Series(y_test).value_counts(normalize=True)}).loc[CLASSES].round(3)

всего для задачи 1: 2930 | train: 2344 | test: 586


,train,test
negative,0.308,0.305
neutral,0.593,0.606
positive,0.098,0.089


## 3. Старые модели на кросс-валидации

Мы лоханулись в первой домашке и не прогоняли модельки первой задачи кросс-валидацией, поэтому исправляемся здесь, чтобы дальше можно было корректно сравнивать метрики новых моделек.

In [6]:
old_model_names = ["hw1_most_frequent", "hw1_stratified", "hw1_bow_logreg", "hw1_tfidf_logreg", "hw1_kirp_model"]
old_models = {name: model for name, model in su.load_models("hw1_").items() if name in old_model_names}

X_train_old = [su.old_dialog_text(d) for d in train_dialogs]

for name, model in old_models.items():
    cv_table = su.save_result("cv_table.csv", name, su.cv_metrics(model, X_train_old, y_train, folds))

cv_table.loc[[m for m in cv_table.index if m.startswith("hw1_")]].round(3)

,macro_f1,accuracy,balanced_accuracy,roc_auc_macro,pr_auc_macro,precision_negative,recall_negative,precision_neutral,recall_neutral,precision_positive,recall_positive
hw1_tfidf_logreg,0.583,0.704,0.560,0.783,0.621,0.637,0.607,0.740,0.830,0.596,0.243
hw1_kirp_model,0.564,0.701,0.542,0.777,0.609,0.637,0.588,0.731,0.843,0.616,0.196
hw1_bow_logreg,0.529,0.656,0.516,0.703,0.514,0.585,0.505,0.710,0.803,0.374,0.239
hw1_stratified,0.348,0.467,0.349,0.513,0.339,0.329,0.357,0.607,0.584,0.109,0.104
hw1_most_frequent,0.248,0.593,0.333,0.500,0.333,0.000,0.000,0.593,1.000,0.000,0.000


## 4. Старые модели на test 

Дабы удостовериться в корректности переноса моделек, на всякий случай посмотри, что покажет тест и сравним с иетриками из 1 домашки.

In [7]:
X_test_old = [su.old_dialog_text(d) for d in test_dialogs]
for name, model in old_models.items():
    test_table = su.save_result("test_table.csv", name, su.all_metrics(y_test, model.predict_proba(X_test_old)))
    
test_table.loc[[m for m in test_table.index if m.startswith("hw1_")]].round(3)

,macro_f1,accuracy,balanced_accuracy,roc_auc_macro,pr_auc_macro,precision_negative,recall_negative,precision_neutral,recall_neutral,precision_positive,recall_positive
hw1_tfidf_logreg,0.593,0.727,0.559,0.776,0.642,0.692,0.564,0.738,0.882,0.750,0.231
hw1_kirp_model,0.581,0.724,0.547,0.781,0.639,0.685,0.570,0.731,0.879,1.000,0.192
hw1_bow_logreg,0.494,0.654,0.484,0.676,0.493,0.608,0.486,0.702,0.811,0.242,0.154
hw1_stratified,0.324,0.452,0.324,0.493,0.331,0.277,0.291,0.606,0.586,0.091,0.096
hw1_most_frequent,0.252,0.606,0.333,0.500,0.333,0.000,0.000,0.606,1.000,0.000,0.000


Мы проверили, все хорошо, чиселки наколупались такие же.